In [2]:
import numpy as np #sayısal işlemler ve lineer cebir için
import pandas as pd #csv dosyalarını okumak, tablo verileriyle çalışmak için


In [3]:
lettuce_df = pd.read_csv('data/lettuce_dataset.csv', encoding='latin-1')
lettuce_df.head() #ilk 5 datayı döndür.

,Plant_ID,Date,Temperature (°C),Humidity (%),TDS Value (ppm),pH Level,Growth Days
0,1,8/3/2023,33.4,53,582,6.4,1
1,1,8/4/2023,33.5,53,451,6.1,2
2,1,8/5/2023,33.4,59,678,6.4,3
3,1,8/6/2023,33.4,68,420,6.4,4
4,1,8/7/2023,33.4,74,637,6.5,5


In [4]:
lettuce_df.describe() #DataFrame’deki sayısal sütunların istatistiksel özetini verir.

,Plant_ID,Temperature (°C),Humidity (%),TDS Value (ppm),pH Level,Growth Days
count,3169.000000,3169.000000,3169.000000,3169.000000,3169.000000,3169.000000
mean,35.441780,28.142222,64.873462,598.045440,6.399211,23.140107
std,20.243433,4.670521,8.988985,115.713047,0.234418,13.075415
min,1.000000,18.000000,50.000000,400.000000,6.000000,1.000000
25%,18.000000,23.600000,57.000000,498.000000,6.200000,12.000000
50%,35.000000,30.200000,65.000000,593.000000,6.400000,23.000000
75%,53.000000,31.500000,73.000000,699.000000,6.600000,34.000000
max,70.000000,33.500000,80.000000,800.000000,6.800000,48.000000


In [5]:
lettuce_df.info() #data hakkında genel bilgi.

<class 'pandas.DataFrame'>
RangeIndex: 3169 entries, 0 to 3168
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Plant_ID          3169 non-null   int64  
 1   Date              3169 non-null   str    
 2   Temperature (°C)  3169 non-null   float64
 3   Humidity (%)      3169 non-null   int64  
 4   TDS Value (ppm)   3169 non-null   int64  
 5   pH Level          3169 non-null   float64
 6   Growth Days       3169 non-null   int64  
dtypes: float64(2), int64(4), str(1)
memory usage: 173.4 KB


In [6]:
print("Missing values: ", lettuce_df.isnull().sum()) #eksik değer (null)
print("Duplicate values: ", lettuce_df.duplicated().sum()) #tekrarlayan satır 

Missing values:  Plant_ID            0
Date                0
Temperature (°C)    0
Humidity (%)        0
TDS Value (ppm)     0
pH Level            0
Growth Days         0
dtype: int64
Duplicate values:  0


In [7]:
#tarih sütunu metin olarak okunmuş olabilir. Date sütununu gerçek tarih formatına çevir.
lettuce_df['Date'] = pd.to_datetime(lettuce_df['Date']) 
lettuce_df.set_index('Date', inplace=True)

lettuce_df.head()

,Plant_ID,Temperature (°C),Humidity (%),TDS Value (ppm),pH Level,Growth Days
Date,,,,,,
2023-08-03,1,33.4,53,582,6.4,1
2023-08-04,1,33.5,53,451,6.1,2
2023-08-05,1,33.4,59,678,6.4,3
2023-08-06,1,33.4,68,420,6.4,4
2023-08-07,1,33.4,74,637,6.5,5


In [8]:
from xgboost import XGBRegressor

# Korelasyon analizi. Bir değişken değişirken diğeri de sistematik olarak değişiyor mu
correlations = lettuce_df.drop('Plant_ID', axis=1).corr()['Growth Days'].sort_values()

X = lettuce_df.drop(['Growth Days', 'Plant_ID'], axis=1)
y = lettuce_df['Growth Days']

# Değişkenlere (feature’lara) ne kadar önem verdiği
xgb_model = XGBRegressor(
    n_estimators=100,  # modelde 100 ağaç kullanılıcak.
    random_state=42,   # sonuçlar tekrar üretilebilir olacak
    objective='reg:squarederror'
)

xgb_model.fit(X, y) # modeli tüm veri seti üzerinde eğitiyoruz. 

# model hangi değişkenlere (feature) daha çok önem veriyor 
feature_importances = pd.Series(
    xgb_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

# DataFrames
correlation_df = pd.DataFrame({
    'Feature': correlations.index,
    'Correlation': correlations.values
})

feature_importances_df = pd.DataFrame({
    'Feature': feature_importances.index,
    'Importance': feature_importances.values
})

# Karşılaştırma tablosu
comparison_df = pd.DataFrame({
    'Correlation': correlations.loc[feature_importances.index].values,
    'Importances': feature_importances.values
},
index=feature_importances.index)

# karşılaştırma tablosunu göster.
comparison_df

,Correlation,Importances
Temperature (°C),-0.074601,0.686147
TDS Value (ppm),-0.020633,0.108621
pH Level,0.003023,0.108023
Humidity (%),-0.014481,0.097210


In [9]:
# Verisetine yeni özellikler eklemek için fonksiyon.
def create_lagged_features(df):
# Sıcaklık, nem, pH ve TDS değerleri için 1, 2, 3 ve 7 gün önceki değerler (lagged features) üretilecek. 
    lag_features = ['Temperature (°C)', 'Humidity (%)', 'pH Level', 'TDS Value (ppm)']
    lags = [1, 2, 3, 7]  # Lags of 1 day, 2 days, 3 days, and 7 days

    for feature in lag_features:
        for lag in lags:
            # Her bitki için ayrı ayrı geçmiş gün değerleri oluşturuyoruz.
            df[f"{feature} Lag {lag}"] = df.groupby('Plant_ID')[feature].shift(lag)

    window = 7  # hareketli pencere (window) boyutu. her hesaplama son 7 satır/veri noktası üzerinden yapılır.
    for feature in lag_features:
        df[f"{feature} Rolling Mean"] = df[feature].rolling(window=window).mean() #hareketli ortalama hesaplama
        df[f"{feature} Rolling Std"] = df[feature].rolling(window=window).std() #hareketli standart sapma hesaplama
    
    # Zaman bazlı (gün, ay) feature
    df['Day of Week'] = df.index.dayofweek + 1
    df['Month'] = df.index.month

#ana veri setine uyguladık.
create_lagged_features(lettuce_df) 


In [10]:
lettuce_df.head()

,Plant_ID,Temperature (°C),Humidity (%),TDS Value (ppm),pH Level,Growth Days,Temperature (°C) Lag 1,Temperature (°C) Lag 2,Temperature (°C) Lag 3,Temperature (°C) Lag 7,...,Temperature (°C) Rolling Mean,Temperature (°C) Rolling Std,Humidity (%) Rolling Mean,Humidity (%) Rolling Std,pH Level Rolling Mean,pH Level Rolling Std,TDS Value (ppm) Rolling Mean,TDS Value (ppm) Rolling Std,Day of Week,Month
Date,,,,,,,,,,,,,,,,,,,,,
2023-08-03,1,33.4,53,582,6.4,1,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,8
2023-08-04,1,33.5,53,451,6.1,2,33.4,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,8
2023-08-05,1,33.4,59,678,6.4,3,33.5,33.4,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6,8
2023-08-06,1,33.4,68,420,6.4,4,33.4,33.5,33.4,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,8
2023-08-07,1,33.4,74,637,6.5,5,33.4,33.4,33.5,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,8


In [11]:
# XGBoost, hata metrikleri ve eğitim-test ayırma
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Lag ve rolling işlemlerinden dolayı oluşan boş satırları (NaN) kaldırıyoruz.
cleaned_df = lettuce_df.dropna()

# Model girdileri ve hedef değişken tekrar ayrılıyor. Bu sefer eklenmiş zaman özellikleri de X içinde var. (80% eğitim, 20% test)
X = cleaned_df.drop(['Growth Days', 'Plant_ID'], axis=1)
y = cleaned_df['Growth Days']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# XGBoost modeli eğitiliyor.
model = xgb.XGBRegressor()
model.fit(X_train, y_train)

# Test verisi üzerinde tahmin yapılıyor.
y_pred = model.predict(X_test)

# Performans Ölçümü
def evaluate_performance(y_test, y_pred):
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    print('Mean Absolute Error: ', mae)
    print('Mean Squared Error: ', mse)
    print('Root Mean Squared Error: ', rmse)
    print('R2 Score: ', r2)

    
evaluate_performance(y_test, y_pred)

Mean Absolute Error:  1.4056894779205322
Mean Squared Error:  8.553234100341797
Root Mean Squared Error:  2.924591270646515
R2 Score:  0.9247319102287292


In [12]:
# Gerçek ve Tahmin Edilen değerleri karşılaştırma.
pd.DataFrame({
    'Actual Values': y_test,
    'Predicted Values': y_pred
}).reset_index(drop=True)

,Actual Values,Predicted Values
0,17,17.329674
1,33,32.971859
2,44,43.950291
3,24,22.090208
4,17,17.578022
...,...,...
531,45,45.167217
532,27,27.128319
533,29,29.071953
534,41,40.801308


In [13]:
X_test.columns #test verisindeki sütunlar. Modelin hangi özelliklerle eğitildiğini gösteriyoruz.

Index(['Temperature (°C)', 'Humidity (%)', 'TDS Value (ppm)', 'pH Level',
       'Temperature (°C) Lag 1', 'Temperature (°C) Lag 2',
       'Temperature (°C) Lag 3', 'Temperature (°C) Lag 7',
       'Humidity (%) Lag 1', 'Humidity (%) Lag 2', 'Humidity (%) Lag 3',
       'Humidity (%) Lag 7', 'pH Level Lag 1', 'pH Level Lag 2',
       'pH Level Lag 3', 'pH Level Lag 7', 'TDS Value (ppm) Lag 1',
       'TDS Value (ppm) Lag 2', 'TDS Value (ppm) Lag 3',
       'TDS Value (ppm) Lag 7', 'Temperature (°C) Rolling Mean',
       'Temperature (°C) Rolling Std', 'Humidity (%) Rolling Mean',
       'Humidity (%) Rolling Std', 'pH Level Rolling Mean',
       'pH Level Rolling Std', 'TDS Value (ppm) Rolling Mean',
       'TDS Value (ppm) Rolling Std', 'Day of Week', 'Month'],
      dtype='str')

In [14]:
# İki marul bitkisinden 15 gün boyunca elde edilen, daha önce görülmemiş (unseen) veriler.
unseen_data = pd.read_csv('data/unseen_data.csv', encoding='latin-1')
unseen_data.shape # Satır-sütun sayısı

(30, 6)

In [15]:
unseen_data.head() #verinin ilk 5 satırı

,Plant_ID,Date,Temperature (°C),Humidity (%),pH Level,TDS Value (ppm)
0,1,9/15/2023,30,60,6.5,500
1,1,9/16/2023,31,62,6.6,505
2,1,9/17/2023,26,58,6.4,495
3,1,9/18/2023,32,57,6.7,490
4,1,9/19/2023,25,59,6.5,500


In [16]:
#tarih sütunu metin olarak okunmuş olabilir. Date sütununu gerçek tarih formatına çevir.
unseen_data['Date'] = pd.to_datetime(unseen_data['Date'])
unseen_data.set_index('Date', inplace=True)

In [17]:
unseen_data.head()

,Plant_ID,Temperature (°C),Humidity (%),pH Level,TDS Value (ppm)
Date,,,,,
2023-09-15,1,30,60,6.5,500
2023-09-16,1,31,62,6.6,505
2023-09-17,1,26,58,6.4,495
2023-09-18,1,32,57,6.7,490
2023-09-19,1,25,59,6.5,500


In [18]:
# yeni veriyi kopyaladık ve daha önce yazılan create_lagged_features fonksiyonu bu veriye de uyguladık.
final_unseen_data = unseen_data.copy()
create_lagged_features(final_unseen_data)

In [19]:
final_unseen_data.head()

,Plant_ID,Temperature (°C),Humidity (%),pH Level,TDS Value (ppm),Temperature (°C) Lag 1,Temperature (°C) Lag 2,Temperature (°C) Lag 3,Temperature (°C) Lag 7,Humidity (%) Lag 1,...,Temperature (°C) Rolling Mean,Temperature (°C) Rolling Std,Humidity (%) Rolling Mean,Humidity (%) Rolling Std,pH Level Rolling Mean,pH Level Rolling Std,TDS Value (ppm) Rolling Mean,TDS Value (ppm) Rolling Std,Day of Week,Month
Date,,,,,,,,,,,,,,,,,,,,,
2023-09-15,1,30,60,6.5,500,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,9
2023-09-16,1,31,62,6.6,505,30.0,NaN,NaN,NaN,60.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6,9
2023-09-17,1,26,58,6.4,495,31.0,30.0,NaN,NaN,62.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,9
2023-09-18,1,32,57,6.7,490,26.0,31.0,30.0,NaN,58.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,9
2023-09-19,1,25,59,6.5,500,32.0,26.0,31.0,NaN,57.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,9


In [20]:
# NaN satırlarını temizledik. 
unseen_data_cleaned = final_unseen_data.dropna()

unseen_data_cleaned.head()

,Plant_ID,Temperature (°C),Humidity (%),pH Level,TDS Value (ppm),Temperature (°C) Lag 1,Temperature (°C) Lag 2,Temperature (°C) Lag 3,Temperature (°C) Lag 7,Humidity (%) Lag 1,...,Temperature (°C) Rolling Mean,Temperature (°C) Rolling Std,Humidity (%) Rolling Mean,Humidity (%) Rolling Std,pH Level Rolling Mean,pH Level Rolling Std,TDS Value (ppm) Rolling Mean,TDS Value (ppm) Rolling Std,Day of Week,Month
Date,,,,,,,,,,,,,,,,,,,,,
2023-09-22,1,26,58,6.7,490,31.0,30.0,25.0,30.0,62.0,...,28.714286,2.927700,59.428571,1.988060,6.571429,0.111270,497.142857,6.362090,5,9
2023-09-23,1,32,57,6.9,525,26.0,31.0,30.0,31.0,58.0,...,28.857143,3.078342,58.714286,1.799471,6.614286,0.167616,500.000000,12.247449,6,9
2023-09-24,1,30,69,6.6,506,32.0,26.0,31.0,26.0,57.0,...,29.428571,2.819997,60.285714,4.231402,6.642857,0.139728,501.571429,12.204605,7,9
2023-09-25,1,31,60,6.5,500,30.0,32.0,26.0,32.0,69.0,...,29.285714,2.690371,60.714286,3.988077,6.614286,0.146385,503.000000,11.165423,1,9
2023-09-26,1,26,62,6.6,505,31.0,30.0,32.0,25.0,60.0,...,29.428571,2.439750,61.142857,3.933979,6.628571,0.138013,503.714286,11.101265,2,9


In [21]:
# unseen_data içinde hangi orijinal sütunların olduğunu gösteriyor.
# henüz lag özellikleri yok; çünkü lag özellikleri final_unseen_data üzerinde oluşturuldu.
unseen_data.columns

Index(['Plant_ID', 'Temperature (°C)', 'Humidity (%)', 'pH Level',
       'TDS Value (ppm)'],
      dtype='str')

In [22]:
# Plant_ID tahmin girdilerinden çıkarılıyor. Kalan sütunlar modele verilecek özellikler.

unseen_features = unseen_data_cleaned.drop('Plant_ID', axis=1)
unseen_features.shape

(16, 30)

In [23]:
# Feature (özellik) sutunları listeleniyor.
unseen_features.columns

Index(['Temperature (°C)', 'Humidity (%)', 'pH Level', 'TDS Value (ppm)',
       'Temperature (°C) Lag 1', 'Temperature (°C) Lag 2',
       'Temperature (°C) Lag 3', 'Temperature (°C) Lag 7',
       'Humidity (%) Lag 1', 'Humidity (%) Lag 2', 'Humidity (%) Lag 3',
       'Humidity (%) Lag 7', 'pH Level Lag 1', 'pH Level Lag 2',
       'pH Level Lag 3', 'pH Level Lag 7', 'TDS Value (ppm) Lag 1',
       'TDS Value (ppm) Lag 2', 'TDS Value (ppm) Lag 3',
       'TDS Value (ppm) Lag 7', 'Temperature (°C) Rolling Mean',
       'Temperature (°C) Rolling Std', 'Humidity (%) Rolling Mean',
       'Humidity (%) Rolling Std', 'pH Level Rolling Mean',
       'pH Level Rolling Std', 'TDS Value (ppm) Rolling Mean',
       'TDS Value (ppm) Rolling Std', 'Day of Week', 'Month'],
      dtype='str')

In [24]:
# Eğitilmiş XGBoost modelini kullanarak unseen veriler için Growth Days tahmin etme

unseen_features = unseen_features[X_train.columns] #yeni verinin sütunlarını eğitim verisindeki sütun sırasına göre yeniden düzenler.
y_unseen_pred = model.predict(unseen_features) #Eğitilmiş ilk modelle yeni verinin Growth Days tahmini yapılır.

predicted_growth_days_df = pd.DataFrame({
    'Plant_ID': unseen_data_cleaned['Plant_ID'],
    'Predicted Growth Days': y_unseen_pred
})

predicted_growth_days_df.reset_index(drop=True)

,Plant_ID,Predicted Growth Days
0,1,41.157162
1,1,37.114552
2,1,44.315788
3,1,40.998138
4,1,41.886883
5,1,35.994778
6,1,41.007530
7,1,42.933296
8,2,41.107639
9,2,42.986427


In [25]:
# Her bitki için büyüme günlerinin ortalaması.
data = predicted_growth_days_df.groupby('Plant_ID')['Predicted Growth Days'].mean().reset_index()

# tabloya dönüştürme
mean_growth_days_per_plant = pd.DataFrame(data)
mean_growth_days_per_plant

,Plant_ID,Predicted Growth Days
0,1,40.676018
1,2,40.626648


In [26]:
# GridSearch ile hiperparametre arama
from sklearn.model_selection import GridSearchCV

# Hiperparametre kombinasyonları tanımladık.
param_grid = {
    'n_estimators': [1500, 2000, 2300],
    'learning_rate': [0.01, 0.03, 0.05],
    'max_depth': [5, 7, 9],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

# 3 katlı cross-validation ile parametreler aranıyor. 
# n_jobs=-1 bilgisayarın uygun tüm işlemcilerini kullanır.
grid_search = GridSearchCV(xgb.XGBRegressor(random_state=42), param_grid, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

# en iyi parametreleri getiriyor
best_params = grid_search.best_params_

best_params

{'colsample_bytree': 0.7,
 'learning_rate': 0.01,
 'max_depth': 7,
 'n_estimators': 2300,
 'subsample': 0.9}

In [27]:
# En iyi parametrelerle XGBoost modeli tekrar eğitiliyor
best_model = xgb.XGBRegressor(**best_params, random_state=42)
best_model.fit(X_train, y_train)

# Yeni model test verisinde değerlendirilir.
y_pred_best = best_model.predict(X_test)

evaluate_performance(y_test, y_pred_best)

Mean Absolute Error:  1.2378045320510864
Mean Squared Error:  7.215664386749268
Root Mean Squared Error:  2.6861988732685576
R2 Score:  0.9365024566650391


In [28]:
#yeni veri üzerinde tahmin yapmak için optimize edilmiş best_model kullanılıyor.
y_unseen_pred = best_model.predict(unseen_features) 

# tahminler tabloya aktarılıyor 
predicted_growth_days_df = pd.DataFrame({
    'Plant_ID': unseen_data_cleaned['Plant_ID'],
    'Predicted Growth Days': y_unseen_pred
})

predicted_growth_days_df.reset_index(drop=True)

# Her bitki için ortalama tahmin hesaplanıyor.
data = predicted_growth_days_df.groupby('Plant_ID')['Predicted Growth Days'].mean().reset_index()

mean_growth_days_per_plant = pd.DataFrame(data)
mean_growth_days_per_plant

,Plant_ID,Predicted Growth Days
0,1,37.699982
1,2,37.534199


In [29]:
# Marulun toplam yetişme süresi
total_growth_days = 45

# Hasada kalan gün = 45 - tahmini büyüme günü
mean_growth_days_per_plant['Predicted Harvest Days'] = (
    total_growth_days - mean_growth_days_per_plant['Predicted Growth Days']
)

# Negatif değer çıkarsa 0 yapıyoruz, çünkü hasat süresi geçmiş olabilir
mean_growth_days_per_plant['Predicted Harvest Days'] = (
    mean_growth_days_per_plant['Predicted Harvest Days']
    .clip(lower=0)
    .round(2)
)

mean_growth_days_per_plant

,Plant_ID,Predicted Growth Days,Predicted Harvest Days
0,1,37.699982,7.30
1,2,37.534199,7.47
